In [1]:
#!/usr/bin/env python3
"""
ne_vs_uhlich.py
===============
Quantitative comparison, in PyTorch, of two finite-support noise-enhanced (NE)
estimators for single-tone frequency estimation.

  Uhlich    S. Uhlich, "Bayes risk reduction of estimators using artificial observation
            noise," IEEE Trans. Signal Process. 63(20):5535-5545, 2015, Sec. III-D, IV-B:
                theta_U(x) = sum_k p_k theta_MAP(x + z_k),
            weights p on the simplex and free locations z_k (his Eqs. (28)-(29)), found
            jointly by minimizing the Monte-Carlo Bayes risk (his Eq. (30)); the
            derivative of theta_MAP with respect to x is his implicit-function result
            (Eq. (36)).  K = 1 is the SR estimator of Chen et al. (2008) ("SR" in his Fig. 5).

  Proposed  DSP2026 manuscript, Eqs. (ane), (erm), (Zconstraint) and Algorithm 1:
                theta_P(x) = (1/K) sum_k theta_MAP(x + z_k),
            sum_k z_k = 0 and ||z_k|| <= beta*sigma_v; symmetric multi-scale initialization
            (Step 1), normalized projected-gradient refinement with retraction and
            backtracking (Step 2), validation gate (Step 3).

  Ablations (optional; they separate the effect of each design choice)
  Uhlich-ZM       Uhlich's free weights plus the zero-mean constraint sum_k p_k z_k = 0
  Proposed-Step1  Algorithm 1 without Step 2 (zeroth-order scale search only)

  References      MAP (= MLE = NLS; grid + Gauss-Newton) and MMSE (quadrature)

Both papers use the same model (the manuscript with D = 1, A = 1):
    x(n) = A cos(n theta) + v(n),  n = 1..N,  v(n) ~ N(0, sigma_v^2),  theta ~ U[0, pi].
All estimators share the same MAP routine, the same training, validation and test sets
(common random numbers), the same K, the same SNR convention and the same evaluation.

Sub-commands
------------
  calibrate  identify the SNR axis of Uhlich's Fig. 5 by comparing MAP and MMSE with the
             values he reports for N = 5 at "5 dB" (MAP 0.41, MMSE 0.29)
  matched    train and test at the same SNR (manuscript protocol, Fig. 2(b)); a list of K
             values gives the K sweep of Fig. 2(a)
  mismatch   train at a few SNRs and test on the whole grid (Uhlich's protocol, Fig. 5)
  all        calibrate + matched + mismatch

Presets
-------
  --preset uhlich  N = 5, Uhlich's SNR axis, SNR grid -10:2.5:20 dB, training SNRs
                   {-10, -2.5, 5, 12.5, 20} dB, K in {1, 4}           (Uhlich's setting)
  --preset dsp     N = 32, gamma = A^2/(2 sigma_v^2), SNR grid -15:2:1 dB, K = 8
                   (manuscript setting)
  Every preset value can be overridden, e.g.  --N 32 --K 4 8 --snrs -9 -7 -5

Assumptions (state them if you report these results)
-----------
  A1  Uhlich does not define his SNR.  His values are reproduced only with
      sigma_v^2 = 2 A^2 10^(-SNR/10), i.e. gamma_manuscript = SNR_Uhlich - 6.02 dB:
      MAP/MMSE = 0.41/0.29 at 5 dB, ~1.6/0.8 at -10 dB, ~1e-3 at 20 dB (`calibrate`).
  A2  Uhlich solved (30) with IPOPT and exact derivatives; his initialization is not
      reported.  Here: softmax weights + L-BFGS (strong Wolfe line search) started from
      the --n-starts best of the SAME symmetric multi-scale candidates that Algorithm 1
      screens in Step 1, so both methods receive the same starting information.
      --uhlich-solver ipopt uses IPOPT (limited-memory Hessian) through cyipopt.
  A3  Not reported in the manuscript, set here: eta_0 (in units of sigma_v), eta_min,
      T_r, M, M', and ||G||_max (max |entry| by default; --grad-scaling max-row).
  A4  MAP: N_theta-point grid + T Gauss-Newton steps (manuscript Eq. (newton)).  The
      derivative d theta/dx uses the full Hessian (exact implicit-function theorem,
      Uhlich Eq. (36)); --ift-hessian gn gives J (J^T J)^(-1).  The curvature is floored
      at --ift-floor times the Gauss-Newton curvature (0 = exact) to avoid gradient spikes.
  A5  The validation gate (Algorithm 1, Step 3) is applied to the Proposed variants only,
      as in the manuscript; --gate-all applies the same gate to every trained method.
      The CSV files always report the un-gated result as well (mse_raw).

Outputs (in --out-dir)
-------
  calibration.csv, matched_N{N}.csv, matched_pairs_N{N}.csv, mismatch_N{N}.csv,
  banks_matched_N{N}.json / banks_mismatch_N{N}.json (Z, p, gate statistics, traces),
  config_*.json and, with matplotlib, matched_N{N}_K{K}.png/.pdf, mismatch_N{N}_K{K}.png/.pdf

Usage
-----
  pip install torch numpy matplotlib          (cyipopt only for --uhlich-solver ipopt)
  python ne_vs_uhlich.py calibrate --preset uhlich
  python ne_vs_uhlich.py matched   --preset uhlich --quick        # smoke test (minutes)
  python ne_vs_uhlich.py all       --preset uhlich                # Uhlich's setting
  python ne_vs_uhlich.py matched   --preset dsp --device cuda     # manuscript setting
  python ne_vs_uhlich.py mismatch  --preset dsp --methods Uhlich Proposed

Runtime (rough estimate, not measured): one full-batch risk evaluation costs M_train*K
MAP solves, roughly 0.5-1 s for N = 5 and 1-3 s for N = 32 on a CPU with M_train = 1e5;
a training run needs ~100-300 evaluations.  `all --preset uhlich` therefore takes a few
CPU hours (the matched protocol is the expensive part), the dsp preset longer; a GPU
(--device cuda) is much faster.  Run with --quick first; --methods Uhlich Proposed and a
smaller --M-train shorten long runs.
"""
from __future__ import annotations

import argparse
import csv
import json
import math
import os
import time
from dataclasses import dataclass
from typing import Any, Callable, Dict, List, Optional, Sequence, Tuple

import numpy as np
import torch

# ======================================================================================
# Constants
# ======================================================================================
PRIOR_VAR = math.pi ** 2 / 12.0

#: sigma_v^2 = factor * A^2 / 10^(SNR_dB / 10)
SNR_CONVENTIONS = {
    "manuscript": 0.5,  # gamma = A^2 / (2 sigma_v^2), per-tone SNR of the manuscript
    "unit": 1.0,        # SNR = A^2 / sigma_v^2
    "uhlich": 2.0,      # axis of Uhlich's Fig. 5 (assumption A1)
}

#: Uhlich (2015), Sec. IV-B: N = 5, banks trained and tested at "5 dB", 30 000 trials
UHLICH_REPORTED_5DB = {"MAP": 0.41, "SR (K=1)": 0.40, "ANE (K=4)": 0.36, "MMSE": 0.29}
#: Manuscript, single-tone section: N = 32, gamma = -7 dB, 1e5 trials
MANUSCRIPT_REPORTED_M7DB = {"MAP": 0.5886, "MMSE": 0.3963, "ANE (K=8)": 0.4247,
                            "ANE (K=10)": 0.4227}

ALL_METHODS = ("Uhlich", "Uhlich-ZM", "Proposed", "Proposed-Step1")

PRESETS: Dict[str, Dict[str, Any]] = {
    "uhlich": dict(
        N=5, snr_convention="uhlich",
        snrs=[-10.0, -7.5, -5.0, -2.5, 0.0, 2.5, 5.0, 7.5, 10.0, 12.5, 15.0, 17.5, 20.0],
        train_snrs=[-10.0, -2.5, 5.0, 12.5, 20.0],
        K=[1, 4], M_train=100_000, M_val=20_000, M_test=100_000,
        quick_snrs=[-2.5, 5.0, 12.5], quick_train_snrs=[5.0]),
    "dsp": dict(
        N=32, snr_convention="manuscript",
        snrs=[-15.0, -13.0, -11.0, -9.0, -7.0, -5.0, -3.0, -1.0, 1.0],
        train_snrs=[-11.0, -7.0, -3.0],
        K=[8], M_train=100_000, M_val=20_000, M_test=100_000,
        quick_snrs=[-11.0, -7.0, -3.0], quick_train_snrs=[-7.0]),
}


def sigma_from_snr(snr_db: float, A: float, convention: str) -> float:
    """Noise standard deviation sigma_v for an SNR in dB under the given convention."""
    return math.sqrt(SNR_CONVENTIONS[convention] * A * A / 10.0 ** (snr_db / 10.0))


def gamma_offset_db(convention: str) -> float:
    """gamma_manuscript [dB] = SNR [dB] + offset, with gamma = A^2 / (2 sigma_v^2)."""
    return -10.0 * math.log10(2.0 * SNR_CONVENTIONS[convention])


# ======================================================================================
# Signal model, MAP and MMSE estimators
# ======================================================================================
class ToneModel:
    """x(n) = A cos(n theta) + v(n), n = 1..N, theta ~ U[0, pi].

    MAP (= MLE = NLS for the uniform prior): grid search of x^T s(phi) - ||s(phi)||^2/2
    over N_theta points of [0, pi] (manuscript Eq. (map)), then T Gauss-Newton (or
    Newton) steps (Eq. (newton)), each limited to one grid spacing and kept in [0, pi];
    the refined value is kept only if the NLS cost does not increase.  The derivative
    with respect to the data follows from the implicit-function theorem,
        d theta / d x(n) = -A n sin(n theta) / H,
    with H the full second derivative of the NLS cost (Uhlich Eq. (36); manuscript
    Eq. (ift)) or its Gauss-Newton approximation.
    """

    def __init__(self, N: int, A: float = 1.0, n_grid: int = 1024, newton_iters: int = 3,
                 refine_hessian: str = "gn", ift_hessian: str = "full",
                 ift_floor: float = 0.05, device: str = "cpu",
                 dtype: torch.dtype = torch.float64, grid_dtype: torch.dtype = torch.float32,
                 chunk: int = 4096):
        self.N, self.A = int(N), float(A)
        self.device = torch.device(device)
        self.dtype, self.grid_dtype = dtype, grid_dtype
        self.chunk = int(chunk)
        self.n_grid = int(n_grid)
        self.newton_iters = int(newton_iters)
        self.refine_hessian, self.ift_hessian = refine_hessian, ift_hessian
        self.ift_floor = float(ift_floor)
        self.n = torch.arange(1, self.N + 1, device=self.device, dtype=dtype)           # (N,)
        self.phi = torch.linspace(0.0, math.pi, self.n_grid, device=self.device, dtype=dtype)
        S = self.A * torch.cos(self.n[:, None] * self.phi[None, :])                      # (N, G)
        self.S_grid = S.to(grid_dtype)
        self.half_energy = (0.5 * (S * S).sum(0)).to(grid_dtype)                          # (G,)
        self.dphi = math.pi / (self.n_grid - 1)
        self.h_floor = 1e-10 * self.A ** 2 * float((self.n ** 2).sum())
        self.edge = 1e-12

    def signal(self, theta: torch.Tensor) -> torch.Tensor:
        """s(theta) for theta of shape (...,), returned with shape (..., N)."""
        return self.A * torch.cos(theta[..., None] * self.n)

    def _terms(self, Xp: torch.Tensor, th: torch.Tensor):
        """NLS cost ||x - s||^2/2, its theta-derivative g, the Gauss-Newton and the full
        second derivatives, and A n sin(n theta) (= -ds/dtheta); Xp (..., N), th (...)."""
        nth = th[..., None] * self.n
        c, s = torch.cos(nth), torch.sin(nth)
        r = Xp - self.A * c
        Ans = self.A * self.n * s
        g = (r * Ans).sum(-1)
        h_gn = (Ans * Ans).sum(-1)
        h_full = h_gn + (r * (self.A * self.n * self.n * c)).sum(-1)
        cost = 0.5 * (r * r).sum(-1)
        return cost, g, h_gn, h_full, Ans

    def _curvature(self, h_gn, h_full, which: str, rel_floor: float):
        h = h_full if which == "full" else h_gn
        h = torch.maximum(h, rel_floor * h_gn)
        return torch.where(h > self.h_floor, h, h_gn.clamp_min(self.h_floor))

    def _refine(self, Xp: torch.Tensor, th0: torch.Tensor) -> torch.Tensor:
        if self.newton_iters <= 0:
            return th0
        th, cost0 = th0, None
        for i in range(self.newton_iters):
            cost, g, h_gn, h_full, _ = self._terms(Xp, th)
            if i == 0:
                cost0 = cost
            h = self._curvature(h_gn, h_full, self.refine_hessian, 0.0)
            th = torch.clamp(th - torch.clamp(g / h, -self.dphi, self.dphi), 0.0, math.pi)
        cost1 = self._terms(Xp, th)[0]
        return torch.where(cost1 <= cost0, th, th0)

    def _dtheta_dx(self, Xp: torch.Tensor, th: torch.Tensor) -> torch.Tensor:
        """Implicit-function derivative (zero when the estimate sits on 0 or pi)."""
        _, _, h_gn, h_full, Ans = self._terms(Xp, th)
        h = self._curvature(h_gn, h_full, self.ift_hessian, self.ift_floor)
        d = -Ans / h[..., None]
        inside = (th > self.edge) & (th < math.pi - self.edge)
        return torch.where(inside[..., None], d, torch.zeros_like(d))

    @torch.no_grad()
    def map_bank(self, X: torch.Tensor, Z: torch.Tensor, want_grad: bool = False):
        """MAP estimates of the K perturbed observations x + z_k.

        X (B, N), Z (K, N) -> theta (B, K) and, if want_grad, d theta / dx (B, K, N).
        The correlation x^T s(phi) is computed once and shared by the K branches;
        Gamma_k = z_k^T s(phi) - ||s(phi)||^2/2 is added per branch (Algorithm 1, Step 5).
        """
        K = Z.shape[0]
        base = X.to(self.grid_dtype) @ self.S_grid                                     # (B, G)
        gam = Z.to(self.grid_dtype) @ self.S_grid - self.half_energy                   # (K, G)
        idx = torch.stack([torch.argmax(base + gam[k], dim=1) for k in range(K)], dim=1)
        Xp = X[:, None, :] + Z[None, :, :]                                             # (B, K, N)
        th = self._refine(Xp, self.phi[idx])
        return th, (self._dtheta_dx(Xp, th) if want_grad else None)

    @torch.no_grad()
    def map_estimate(self, X: torch.Tensor) -> torch.Tensor:
        Z0 = torch.zeros(1, self.N, device=X.device, dtype=X.dtype)
        return torch.cat([self.map_bank(X[i:i + self.chunk], Z0)[0][:, 0]
                          for i in range(0, X.shape[0], self.chunk)])

    @torch.no_grad()
    def mmse_estimate(self, X: torch.Tensor, sigma: float, n_quad: int = 4096) -> torch.Tensor:
        """Posterior mean by trapezoidal quadrature on n_quad points of [0, pi]."""
        phi = torch.linspace(0.0, math.pi, n_quad, device=X.device, dtype=self.dtype)
        S = self.A * torch.cos(self.n[:, None] * phi[None, :])
        half = 0.5 * (S * S).sum(0)
        logw = torch.zeros(n_quad, device=X.device, dtype=self.dtype)
        logw[0] = logw[-1] = math.log(0.5)
        rows = max(64, (self.chunk * 1024) // n_quad)
        out = []
        for i in range(0, X.shape[0], rows):
            ll = (X[i:i + rows] @ S - half) / (sigma * sigma) + logw
            out.append(torch.softmax(ll, dim=1) @ phi)
        return torch.cat(out)


# ======================================================================================
# Data (common random numbers: x = s(theta) + sigma_v V for every SNR and every method)
# ======================================================================================
@dataclass
class Split:
    theta: torch.Tensor   # (M,)
    s: torch.Tensor       # (M, N) noiseless signals
    V: torch.Tensor       # (M, N) standard normal noise

    def X(self, sigma: float) -> torch.Tensor:
        return self.s + sigma * self.V


def make_split(model: ToneModel, M: int, seed: int) -> Split:
    g = torch.Generator().manual_seed(int(seed))
    theta = torch.rand(M, generator=g, dtype=torch.float64) * math.pi
    V = torch.randn(M, model.N, generator=g, dtype=torch.float64)
    theta = theta.to(device=model.device, dtype=model.dtype)
    V = V.to(device=model.device, dtype=model.dtype)
    return Split(theta, model.signal(theta), V)


# ======================================================================================
# Finite-support NE estimator, empirical risk and its gradient
# ======================================================================================
@torch.no_grad()
def bank_estimate(model: ToneModel, X: torch.Tensor, Z: torch.Tensor,
                  p: torch.Tensor) -> torch.Tensor:
    """sum_k p_k theta_MAP(x + z_k) for every row of X."""
    return torch.cat([model.map_bank(X[i:i + model.chunk], Z)[0] @ p
                      for i in range(0, X.shape[0], model.chunk)])


@dataclass
class Budget:
    n_risk: int = 0       # full-batch risk evaluations (no gradient)
    n_grad: int = 0       # full-batch risk + gradient evaluations
    seconds: float = 0.0


def bank_risk(model: ToneModel, X: torch.Tensor, theta: torch.Tensor, Z: torch.Tensor,
              p: torch.Tensor, budget: Optional[Budget] = None) -> float:
    """Empirical Bayes risk (manuscript Eq. (erm); Uhlich Eq. (30a))."""
    if budget is not None:
        budget.n_risk += 1
    return float(((bank_estimate(model, X, Z, p) - theta) ** 2).mean())


class _MAPBank(torch.autograd.Function):
    """theta_MAP(x + z_k) for all k, differentiated with the implicit-function theorem.

    This is the pathwise derivative on the current branch (peak).  Jumps of theta_MAP
    between peaks contribute nothing to it; the same holds for Uhlich's Eq. (36) and for
    the manuscript's Eq. (grad)."""

    @staticmethod
    def forward(ctx, X, Z, model):
        th, dtdx = model.map_bank(X, Z, want_grad=True)
        ctx.save_for_backward(dtdx)
        return th

    @staticmethod
    def backward(ctx, grad_th):
        (dtdx,) = ctx.saved_tensors
        return None, torch.einsum("bk,bkn->kn", grad_th, dtdx), None


def risk_and_grad(model: ToneModel, X: torch.Tensor, theta: torch.Tensor,
                  bank_fn: Callable, params: Sequence[torch.Tensor], budget: Budget) -> float:
    """Full-batch empirical risk; gradients are accumulated into params[i].grad.
    bank_fn() returns (Z, p), differentiable with respect to params."""
    for t in params:
        t.grad = None
    M, total = X.shape[0], 0.0
    for i in range(0, M, model.chunk):
        Z, p = bank_fn()
        th = _MAPBank.apply(X[i:i + model.chunk], Z, model)
        loss = ((th @ p - theta[i:i + model.chunk]) ** 2).sum() / M
        loss.backward()
        total += float(loss.detach())
    budget.n_grad += 1
    return total


def bank_directions(K: int, N: int, seed: int) -> torch.Tensor:
    """Algorithm 1, Step 1: U = [u_1, -u_1, ..., u_{K/2}, -u_{K/2}], ||u_k|| = 1.
    For odd K (Uhlich's method only) one extra random unit direction is appended."""
    g = torch.Generator().manual_seed(int(seed))
    rows: List[torch.Tensor] = []
    for _ in range(K // 2):
        u = torch.randn(N, generator=g, dtype=torch.float64)
        u = u / u.norm()
        rows += [u, -u]
    if K % 2:
        u = torch.randn(N, generator=g, dtype=torch.float64)
        rows.append(u / u.norm())
    return torch.stack(rows)


# --------------------------------------------------------------------------------------
# Uhlich (2015): free weights and free locations, Eq. (30)
# --------------------------------------------------------------------------------------
class FreeBank:
    """Weights p = softmax(a) (simplex, Uhlich Eq. (30b)) and locations z_k = sigma_v y_k.
    zero_mean=True imposes sum_k p_k z_k = 0 exactly via z_k = sigma_v (y_k - sum_j p_j y_j)
    (ablation 'Uhlich-ZM')."""

    def __init__(self, Y0: torch.Tensor, sigma: float, zero_mean: bool = False):
        K = Y0.shape[0]
        self.Y = Y0.detach().clone().requires_grad_(True)
        self.a = torch.zeros(K, dtype=Y0.dtype, device=Y0.device, requires_grad=K > 1)
        self.sigma, self.zero_mean = float(sigma), bool(zero_mean)

    @property
    def params(self) -> List[torch.Tensor]:
        return [self.Y, self.a] if self.a.requires_grad else [self.Y]

    def __call__(self):
        p = torch.softmax(self.a, dim=0)
        Y = self.Y
        if self.zero_mean:
            Y = Y - (p[:, None] * Y).sum(0, keepdim=True)
        return self.sigma * Y, p


def fit_lbfgs(model: ToneModel, X: torch.Tensor, theta: torch.Tensor, bank: FreeBank,
              max_iter: int, history: int, budget: Budget):
    opt = torch.optim.LBFGS(bank.params, lr=1.0, max_iter=max_iter,
                            max_eval=int(1.25 * max_iter) + 1, history_size=history,
                            tolerance_grad=1e-12, tolerance_change=1e-14,
                            line_search_fn="strong_wolfe")

    def closure():
        opt.zero_grad()
        val = risk_and_grad(model, X, theta, bank, bank.params, budget)
        return torch.tensor(val, dtype=X.dtype, device=X.device)

    opt.step(closure)
    with torch.no_grad():
        Z, p = bank()
    return Z.detach().clone(), p.detach().clone()


def fit_ipopt(model: ToneModel, X: torch.Tensor, theta: torch.Tensor, Y0: torch.Tensor,
              sigma: float, max_iter: int, budget: Budget):
    """Uhlich's problem (30) solved with IPOPT, as in his paper (limited-memory Hessian
    instead of the exact one).  Needs `pip install cyipopt`."""
    from cyipopt import minimize_ipopt  # optional dependency

    K, N = Y0.shape
    nY = K * N
    cache: Dict[str, object] = {}

    def evaluate(w: np.ndarray):
        key = w.tobytes()
        if cache.get("key") != key:
            Y = torch.tensor(w[:nY].reshape(K, N), dtype=X.dtype, device=X.device,
                             requires_grad=True)
            p = torch.tensor(w[nY:], dtype=X.dtype, device=X.device, requires_grad=True)
            f = risk_and_grad(model, X, theta, lambda: (sigma * Y, p), [Y, p], budget)
            gY = Y.grad.detach().cpu().numpy().ravel()
            gp = p.grad.detach().cpu().numpy() if p.grad is not None else np.zeros(K)
            cache.update(key=key, f=f, g=np.concatenate([gY, gp]))
        return cache["f"], cache["g"]

    big = 1e20
    w0 = np.concatenate([Y0.detach().cpu().numpy().ravel(), np.full(K, 1.0 / K)])
    res = minimize_ipopt(
        lambda w: evaluate(w)[0], w0, jac=lambda w: evaluate(w)[1],
        bounds=[(-big, big)] * nY + [(0.0, 1.0)] * K,
        constraints=[{"type": "eq",
                      "fun": lambda w: np.array([w[nY:].sum() - 1.0]),
                      "jac": lambda w: np.concatenate([np.zeros(nY), np.ones(K)])[None, :]}],
        options={"max_iter": int(max_iter), "hessian_approximation": "limited-memory",
                 "print_level": 0, "tol": 1e-9})
    w = np.asarray(res.x, dtype=float)
    Z = sigma * torch.tensor(w[:nY].reshape(K, N), dtype=X.dtype, device=X.device)
    p = torch.tensor(np.clip(w[nY:], 0.0, None), dtype=X.dtype, device=X.device)
    return Z, p / p.sum()


def train_uhlich(model: ToneModel, Xtr: torch.Tensor, ttr: torch.Tensor, sigma: float,
                 K: int, U: torch.Tensor, args, budget: Budget, zero_mean: bool = False):
    """Uhlich's formulation: min over (p, z_1..z_K) of the empirical risk, p on the simplex.
    Starts: the --n-starts best (zeroth-order screen, uniform weights) of the scales
    alpha*sigma_v*U also used by Algorithm 1 (plus z = 0 when K = 1)."""
    U = U.to(device=model.device, dtype=model.dtype)
    p_uniform = torch.full((K,), 1.0 / K, dtype=model.dtype, device=model.device)
    scales = ([0.0] if K == 1 else []) + [float(a) for a in args.alphas]
    screen = []
    for a in scales:
        Y = a * U
        if zero_mean:
            Y = Y - Y.mean(0, keepdim=True)
        screen.append(bank_risk(model, Xtr, ttr, sigma * Y, p_uniform, budget))
    best = None
    for j in [int(i) for i in np.argsort(screen)[:max(1, args.n_starts)]]:
        Y0 = scales[j] * U
        Z = p = None
        if args.uhlich_solver == "ipopt" and not zero_mean:
            try:
                Z, p = fit_ipopt(model, Xtr, ttr, Y0, sigma, args.ipopt_iter, budget)
            except Exception as exc:  # cyipopt missing or solver failure
                print(f"    [Uhlich] IPOPT unavailable or failed ({exc!r}); using L-BFGS")
                Z = p = None
        if Z is None:
            bank = FreeBank(Y0, sigma, zero_mean)
            Z, p = fit_lbfgs(model, Xtr, ttr, bank, args.lbfgs_iter, args.lbfgs_history, budget)
        loss = bank_risk(model, Xtr, ttr, Z, p, budget)
        if best is None or loss < best[0]:
            best = (loss, Z, p, scales[j])
    assert best is not None
    loss, Z, p, a0 = best
    trace = {"train_loss": loss, "start_scale": a0, "screen_scales": scales,
             "screen_losses": screen}
    return Z, p, trace


# --------------------------------------------------------------------------------------
# Proposed: Algorithm 1 of the manuscript
# --------------------------------------------------------------------------------------
def retract(Z: torch.Tensor, radius: float) -> torch.Tensor:
    """Projection onto the zero-sum subspace followed by the common rescaling
    S(Z) = Z0 * min(1, beta*sigma_v / max_k ||z0_k||) (keeps the zero-sum property)."""
    Z0 = Z - Z.mean(0, keepdim=True)
    m = float(Z0.norm(dim=1).max())
    if radius > 0 and m > radius:
        Z0 = Z0 * (radius / m)
    return Z0


def train_proposed(model: ToneModel, Xtr: torch.Tensor, ttr: torch.Tensor, sigma: float,
                   K: int, U: torch.Tensor, args, budget: Budget, refine: bool = True):
    N = model.N
    radius = args.beta * sigma
    pK = torch.full((K,), 1.0 / K, dtype=model.dtype, device=model.device)
    U = U.to(device=model.device, dtype=model.dtype)
    # Step 1: zero bank and S(alpha sigma_v U), alpha in E_alpha; keep the best on training data
    names = ["zero"] + [f"alpha={a:g}" for a in args.alphas]
    cands = [torch.zeros(K, N, dtype=model.dtype, device=model.device)]
    cands += [retract(float(a) * sigma * U, radius) for a in args.alphas]
    losses = [bank_risk(model, Xtr, ttr, Zc, pK, budget) for Zc in cands]
    j = int(np.argmin(losses))
    Z, L = cands[j], losses[j]
    # Step 2: normalized projected gradient + retraction + backtracking
    eta, accepted_steps = float(args.eta0), 0
    if refine and float(Z.abs().max()) > 0.0:   # Z = 0 is a stationary point (manuscript)
        for _ in range(args.T_r):
            Zv = Z.clone().requires_grad_(True)
            L_cur = risk_and_grad(model, Xtr, ttr, lambda Zv=Zv: (Zv, pK), [Zv], budget)
            G = Zv.grad.detach()
            G = G - G.mean(0, keepdim=True)          # projection onto the zero-sum subspace
            gmax = float(G.norm(dim=1).max()) if args.grad_scaling == "max-row" \
                else float(G.abs().max())
            if gmax == 0.0:
                break
            accepted, Zt, Lt = False, Z, L
            while eta >= args.eta_min:
                Zt = retract(Z - (eta * sigma / gmax) * G, radius)
                Lt = bank_risk(model, Xtr, ttr, Zt, pK, budget)
                if Lt < L_cur:
                    accepted = True
                    break
                eta *= 0.5
            if not accepted:
                break
            Z, L, eta = Zt, Lt, eta * 1.3
            accepted_steps += 1
    trace = {"train_loss": L, "step1_choice": names[j], "step1_losses": losses,
             "accepted_steps": accepted_steps}
    return Z.detach(), pK, trace


def validation_gate(model: ToneModel, Xv: torch.Tensor, tv: torch.Tensor,
                    e2_map_val: torch.Tensor, Z: torch.Tensor, p: torch.Tensor,
                    Delta: float, kappa: float, t_thr: float) -> dict:
    """Algorithm 1, Step 3: paired t statistic, McNemar statistic for anomaly transitions
    across Delta = 2 pi / N, and relative gain G; accept if (T_t < -3 or T_mc > 3), G >= kappa."""
    e2 = (bank_estimate(model, Xv, Z, p) - tv) ** 2
    d = e2 - e2_map_val
    sd = float(d.std())
    T_t = float(d.mean()) * math.sqrt(d.numel()) / sd if sd > 0 else 0.0
    e, e0 = e2.sqrt(), e2_map_val.sqrt()
    n10 = int(((e0 > Delta) & (e <= Delta)).sum())
    n01 = int(((e0 <= Delta) & (e > Delta)).sum())
    T_mc = (n10 - n01) / math.sqrt(n10 + n01) if n10 + n01 > 0 else 0.0
    G = 1.0 - float(e2.sum() / e2_map_val.sum())
    accept = bool((T_t < -t_thr or T_mc > t_thr) and G >= kappa)
    return {"accept": accept, "T_t": T_t, "T_mc": T_mc, "val_gain": G, "n10": n10, "n01": n01}


# ======================================================================================
# Training dispatcher and evaluation
# ======================================================================================
@dataclass
class TrainedBank:
    method: str
    K: int
    Z: torch.Tensor
    p: torch.Tensor
    snr_train: float
    sigma_train: float
    budget: Budget
    trace: dict
    gate: dict
    gate_used: bool

    @property
    def deployed(self) -> bool:
        """False if the validation gate rejected the bank (then Z* = 0, i.e. the MAP)."""
        return (not self.gate_used) or bool(self.gate["accept"])


def applicable(method: str, K: int) -> Tuple[bool, str]:
    if method in ("Proposed", "Proposed-Step1") and (K < 2 or K % 2):
        return False, "Algorithm 1 needs an even K >= 2 (symmetric zero-sum pairs)"
    if method == "Uhlich-ZM" and K < 2:
        return False, "for K = 1 the zero-mean bank is z = 0, i.e. the MAP estimator"
    return K >= 1, "K must be >= 1"


def method_label(method: str, K: int) -> str:
    return "SR (Uhlich, K=1)" if (method == "Uhlich" and K == 1) else f"{method} (K={K})"


def train_method(method: str, K: int, model: ToneModel, data: dict, sigma: float,
                 snr: float, U: torch.Tensor, args) -> TrainedBank:
    budget = Budget()
    t0 = time.perf_counter()
    Xtr, ttr = data["Xtr"], data["ttr"]
    if method == "Uhlich":
        Z, p, trace = train_uhlich(model, Xtr, ttr, sigma, K, U, args, budget)
    elif method == "Uhlich-ZM":
        Z, p, trace = train_uhlich(model, Xtr, ttr, sigma, K, U, args, budget, zero_mean=True)
    elif method == "Proposed":
        Z, p, trace = train_proposed(model, Xtr, ttr, sigma, K, U, args, budget, refine=True)
    elif method == "Proposed-Step1":
        Z, p, trace = train_proposed(model, Xtr, ttr, sigma, K, U, args, budget, refine=False)
    else:
        raise ValueError(f"unknown method {method!r}")
    budget.seconds = time.perf_counter() - t0
    gate = validation_gate(model, data["Xv"], data["tv"], data["e2_map_val"], Z, p,
                           args.Delta, args.kappa, args.t_threshold)
    gate_used = (method.startswith("Proposed") and not args.no_gate) or args.gate_all
    return TrainedBank(method, K, Z, p, snr, sigma, budget, trace, gate, gate_used)


def training_data(model: ToneModel, splits: Dict[str, Split], sigma: float) -> dict:
    Xv = splits["val"].X(sigma)
    return {"Xtr": splits["train"].X(sigma), "ttr": splits["train"].theta,
            "Xv": Xv, "tv": splits["val"].theta,
            "e2_map_val": (model.map_estimate(Xv) - splits["val"].theta) ** 2}


def reference_errors(model: ToneModel, split: Split, sigma: float, n_quad: int):
    X = split.X(sigma)
    e2_map = (model.map_estimate(X) - split.theta) ** 2
    e2_mmse = (model.mmse_estimate(X, sigma, n_quad) - split.theta) ** 2
    return X, e2_map, e2_mmse


def test_errors(model: ToneModel, bank: TrainedBank, X: torch.Tensor, theta: torch.Tensor,
                e2_map: torch.Tensor):
    """Squared test errors of the deployed estimator and of the raw (un-gated) bank."""
    e2_raw = (bank_estimate(model, X, bank.Z, bank.p) - theta) ** 2
    return (e2_raw if bank.deployed else e2_map), e2_raw


def describe(e2: torch.Tensor, mse_map: float, mse_mmse: float, Delta: float) -> dict:
    mse = float(e2.mean())
    gap = 100.0 * (mse_map - mse) / (mse_map - mse_mmse) if mse_map > mse_mmse else float("nan")
    return dict(mse=mse, se=float(e2.std() / math.sqrt(e2.numel())),
                gain_vs_map_pct=100.0 * (1.0 - mse / mse_map), gap_closed_pct=gap,
                outlier_rate=float((e2 > Delta * Delta).double().mean()))


def bank_stats(bank: TrainedBank) -> dict:
    Z, p, s = bank.Z, bank.p, bank.sigma_train
    norms = Z.norm(dim=1) / s
    return dict(max_norm_over_sigma=float(norms.max()), min_norm_over_sigma=float(norms.min()),
                mean_shift_over_sigma=float((p[:, None] * Z).sum(0).norm() / s),
                min_weight=float(p.min()), max_weight=float(p.max()))


def bank_row(bank: TrainedBank) -> dict:
    return dict(gate_used=bank.gate_used, gate_accept=bank.gate["accept"],
                deployed=bank.deployed, T_t=bank.gate["T_t"], T_mc=bank.gate["T_mc"],
                val_gain=bank.gate["val_gain"],
                train_loss=bank.trace.get("train_loss", float("nan")),
                train_seconds=bank.budget.seconds, n_risk_evals=bank.budget.n_risk,
                n_grad_evals=bank.budget.n_grad, **bank_stats(bank))


def bank_record(bank: TrainedBank, protocol: str) -> dict:
    return dict(protocol=protocol, method=bank.method, K=bank.K, snr_train_db=bank.snr_train,
                sigma_train=bank.sigma_train, Z=bank.Z.detach().cpu().tolist(),
                p=bank.p.detach().cpu().tolist(), gate=bank.gate, gate_used=bank.gate_used,
                deployed=bank.deployed, trace=bank.trace, stats=bank_stats(bank),
                budget=dict(n_risk=bank.budget.n_risk, n_grad=bank.budget.n_grad,
                            seconds=bank.budget.seconds))


def online_ops(N: int, n_grid: int, K: int, T: int, D: int = 1) -> int:
    """Online cost of manuscript Table 1: MAP for K = 0, a K-point bank otherwise (the
    same for Uhlich's weighted average up to K multiplications)."""
    per_branch = n_grid + T * (N * D * D + D ** 3)
    return N * n_grid + (per_branch if K == 0 else K * per_branch)


def paired_tests(errs: Dict[str, torch.Tensor], e2_map: torch.Tensor, base: dict,
                 K: int) -> List[dict]:
    """Paired comparisons on the same test samples (normal approximation)."""
    comps = [(m, "MAP", errs[m], e2_map) for m in errs]
    for a, b in (("Proposed", "Uhlich"), ("Uhlich-ZM", "Uhlich"),
                 ("Proposed", "Proposed-Step1"), ("Proposed", "Uhlich-ZM")):
        if a in errs and b in errs:
            comps.append((a, b, errs[a], errs[b]))
    out = []
    for a, b, ea, eb in comps:
        d = ea - eb
        m, se = float(d.mean()), float(d.std() / math.sqrt(d.numel()))
        t = m / se if se > 0 else 0.0
        out.append({**base, "K": K, "A": a, "B": b, "mse_A_minus_B": m, "se": se, "t": t,
                    "p_two_sided": math.erfc(abs(t) / math.sqrt(2.0))})
    return out


def print_row(r: dict) -> None:
    gap = r["gap_closed_pct"]
    gap_s = "   n/a" if gap != gap else f"{gap:6.1f}"
    extra = ""
    if "train_seconds" in r:
        extra = (f"  | train {r['train_seconds']:.1f} s, {r['n_risk_evals']} f + "
                 f"{r['n_grad_evals']} g evals, max||z||/sigma {r['max_norm_over_sigma']:.2f}, "
                 f"||sum p z||/sigma {r['mean_shift_over_sigma']:.2f}")
        if r["gate_used"]:
            extra += ", gate " + ("accepted" if r["gate_accept"] else "REJECTED -> MAP")
    print(f"  {r['label']:<20s} MSE {r['mse']:.4g} (se {r['se']:.1g})  "
          f"vs MAP {r['gain_vs_map_pct']:+6.1f}%  gap closed {gap_s}%  "
          f"outliers {100.0 * r['outlier_rate']:5.2f}%{extra}")


def print_reported(args, N: int, snr_train: float, snr_test: float) -> None:
    if N == 5 and args.snr_convention == "uhlich" and snr_train == 5.0 and snr_test == 5.0:
        print("  [reference] Uhlich (2015), N = 5, 5 dB, 30 000 trials: "
              + ", ".join(f"{k} {v}" for k, v in UHLICH_REPORTED_5DB.items()))
    if N == 32 and args.snr_convention == "manuscript" and snr_train == -7.0 and snr_test == -7.0:
        print("  [reference] manuscript, N = 32, -7 dB: "
              + ", ".join(f"{k} {v}" for k, v in MANUSCRIPT_REPORTED_M7DB.items()))


# ======================================================================================
# Protocols
# ======================================================================================
def run_calibrate(args, model: ToneModel, test: Split) -> List[dict]:
    print("\n[calibrate] Uhlich (2015), N = 5: MAP 0.41 and MMSE 0.29 at '5 dB' (text); "
          "about 1.6 / 0.8 at -10 dB and 1e-3 at 20 dB (Fig. 5)")
    if model.N != 5:
        print(f"  note: N = {model.N}, while Uhlich's values refer to N = 5")
    rows, score = [], {}
    for conv in SNR_CONVENTIONS:
        for snr in args.calib_snrs:
            sigma = sigma_from_snr(snr, args.A, conv)
            _, e2_map, e2_mmse = reference_errors(model, test, sigma, args.n_quad)
            m_map, m_mmse = float(e2_map.mean()), float(e2_mmse.mean())
            rows.append(dict(protocol="calibrate", N=model.N, snr_convention=conv,
                             snr_test_db=snr, sigma_test=sigma, mse_map=m_map,
                             se_map=float(e2_map.std() / math.sqrt(e2_map.numel())),
                             mse_mmse=m_mmse,
                             se_mmse=float(e2_mmse.std() / math.sqrt(e2_mmse.numel()))))
            print(f"  {conv:<10s} {snr:+6.1f} dB  sigma_v^2 = {sigma ** 2:9.4f}  "
                  f"MAP {m_map:.4f}  MMSE {m_mmse:.4f}")
            if snr == 5.0:
                score[conv] = abs(math.log(m_map / 0.41)) + abs(math.log(m_mmse / 0.29))
    if score and model.N == 5:
        best = min(score, key=lambda c: score[c])
        print(f"  closest to Uhlich's 5 dB values: '{best}' "
              f"(sigma_v^2 = {SNR_CONVENTIONS[best]:g} A^2 10^(-SNR/10); "
              f"gamma_manuscript = SNR {gamma_offset_db(best):+.2f} dB)")
    return rows


def run_matched(args, model: ToneModel, splits: Dict[str, Split]):
    rows, pairs, banks = [], [], []
    test = splits["test"]
    for snr in args.snrs:
        sigma = sigma_from_snr(snr, args.A, args.snr_convention)
        data = training_data(model, splits, sigma)
        Xte, e2_map, e2_mmse = reference_errors(model, test, sigma, args.n_quad)
        mse_map, mse_mmse = float(e2_map.mean()), float(e2_mmse.mean())
        base = dict(protocol="matched", N=model.N, snr_convention=args.snr_convention,
                    snr_train_db=snr, snr_test_db=snr, sigma_train=sigma, sigma_test=sigma)
        print(f"\n[matched] SNR {snr:+.2f} dB ({args.snr_convention}; manuscript gamma "
              f"{snr + gamma_offset_db(args.snr_convention):+.2f} dB), sigma_v = {sigma:.4f}, "
              f"prior variance {PRIOR_VAR:.4f}")
        for name, e2 in (("MAP", e2_map), ("MMSE", e2_mmse)):
            r = {**base, "method": name, "K": 0, "label": name,
                 **describe(e2, mse_map, mse_mmse, args.Delta)}
            if name == "MAP":
                r["online_ops"] = online_ops(model.N, args.n_grid, 0, args.newton_iters)
            rows.append(r)
            print_row(r)
        for K in args.K:
            U = bank_directions(K, model.N, args.seed + 1000 + K)   # same U for all methods
            errs: Dict[str, torch.Tensor] = {}
            for method in args.methods_matched:
                if not applicable(method, K)[0]:
                    continue
                bank = train_method(method, K, model, data, sigma, snr, U, args)
                e2, e2_raw = test_errors(model, bank, Xte, test.theta, e2_map)
                errs[method] = e2
                r = {**base, "method": method, "K": K, "label": method_label(method, K),
                     **describe(e2, mse_map, mse_mmse, args.Delta),
                     "mse_raw": float(e2_raw.mean()), **bank_row(bank),
                     "online_ops": online_ops(model.N, args.n_grid, K, args.newton_iters)}
                rows.append(r)
                banks.append(bank_record(bank, "matched"))
                print_row(r)
            pairs += paired_tests(errs, e2_map, base, K)
        print_reported(args, model.N, snr, snr)
    return rows, pairs, banks


def run_mismatch(args, model: ToneModel, splits: Dict[str, Split]):
    rows, banks = [], []
    test = splits["test"]
    refs = {}
    for snr in args.snrs:
        sigma = sigma_from_snr(snr, args.A, args.snr_convention)
        _, e2_map, e2_mmse = reference_errors(model, test, sigma, args.n_quad)
        refs[snr] = (sigma, e2_map, e2_mmse)
        for name, e2 in (("MAP", e2_map), ("MMSE", e2_mmse)):
            rows.append(dict(protocol="mismatch", N=model.N, snr_convention=args.snr_convention,
                             snr_train_db="", snr_test_db=snr, sigma_test=sigma, method=name,
                             K=0, label=name, **describe(e2, float(e2_map.mean()),
                                                         float(e2_mmse.mean()), args.Delta)))
    for K in args.K:
        U = bank_directions(K, model.N, args.seed + 1000 + K)
        for snr_tr in args.train_snrs:
            sigma_tr = sigma_from_snr(snr_tr, args.A, args.snr_convention)
            data = training_data(model, splits, sigma_tr)
            print(f"\n[mismatch] K = {K}, trained at {snr_tr:+.2f} dB (sigma_v = {sigma_tr:.4f}); "
                  f"test MSE per test SNR:")
            for method in args.methods_mismatch:
                if not applicable(method, K)[0]:
                    continue
                bank = train_method(method, K, model, data, sigma_tr, snr_tr, U, args)
                banks.append(bank_record(bank, "mismatch"))
                line = []
                for snr_te in args.snrs:
                    sigma_te, e2_map, e2_mmse = refs[snr_te]
                    e2, e2_raw = test_errors(model, bank, test.X(sigma_te), test.theta, e2_map)
                    d = describe(e2, float(e2_map.mean()), float(e2_mmse.mean()), args.Delta)
                    rows.append(dict(protocol="mismatch", N=model.N,
                                     snr_convention=args.snr_convention, snr_train_db=snr_tr,
                                     snr_test_db=snr_te, sigma_train=sigma_tr,
                                     sigma_test=sigma_te, method=method, K=K,
                                     label=method_label(method, K), **d,
                                     mse_raw=float(e2_raw.mean()), **bank_row(bank)))
                    line.append(f"{snr_te:+g}:{d['mse']:.3g}")
                gate = "" if not bank.gate_used else \
                    (" [gate accepted]" if bank.deployed else " [gate rejected -> MAP]")
                print(f"  {method_label(method, K)}{gate}\n    " + "  ".join(line))
            print_reported(args, model.N, snr_tr, snr_tr)
    return rows, banks


def print_matched_summary(rows: List[dict], pairs: List[dict], K_list: Sequence[int]) -> None:
    for K in K_list:
        sel = [r for r in rows if r["protocol"] == "matched" and r["K"] in (0, K)]
        labels: List[str] = []
        for r in sel:
            if r["label"] not in labels:
                labels.append(r["label"])
        if len(labels) <= 2:
            continue
        print(f"\nSummary, matched protocol, K = {K}: test MSE")
        print("  SNR [dB]" + "".join(f"{lab:>22s}" for lab in labels))
        for snr in sorted({r["snr_test_db"] for r in sel}):
            vals = {r["label"]: r["mse"] for r in sel if r["snr_test_db"] == snr}
            print(f"  {snr:+8.2f}" + "".join(f"{vals.get(lab, float('nan')):>22.4g}"
                                             for lab in labels))
        sel_p = [q for q in pairs if q["K"] == K and q["A"] == "Proposed" and q["B"] == "Uhlich"]
        if sel_p:
            print("  Proposed - Uhlich, paired on the same test samples:")
            for q in sel_p:
                print(f"    {q['snr_test_db']:+8.2f} dB: {q['mse_A_minus_B']:+.4g} "
                      f"(se {q['se']:.2g}, t = {q['t']:+.2f}, p = {q['p_two_sided']:.2g})")


# ======================================================================================
# Output and plots
# ======================================================================================
def write_csv(path: str, rows: List[dict]) -> None:
    if not rows:
        return
    keys: List[str] = []
    for r in rows:
        for k in r:
            if k not in keys:
                keys.append(k)
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=keys)
        w.writeheader()
        for r in rows:
            w.writerow({k: r.get(k, "") for k in keys})
    print(f"wrote {path}")


def save_json(path: str, obj) -> None:
    with open(path, "w") as f:
        json.dump(obj, f, indent=1, default=str)
    print(f"wrote {path}")


STYLE = {"MAP": dict(color="tab:red", ls="-", marker=None),
         "MMSE": dict(color="k", ls="-.", marker=None),
         "Uhlich": dict(color="tab:blue", ls="--", marker="o"),
         "Uhlich-ZM": dict(color="tab:cyan", ls=":", marker="v"),
         "Proposed": dict(color="tab:green", ls="-", marker="s"),
         "Proposed-Step1": dict(color="tab:olive", ls=":", marker="^")}
TRAIN_COLORS = ["tab:purple", "tab:blue", "tab:green", "tab:orange", "tab:brown",
                "tab:pink", "tab:gray", "tab:olive", "tab:cyan"]


def _pyplot():
    try:
        import matplotlib
        matplotlib.use("Agg")
        import matplotlib.pyplot as plt
        return plt
    except Exception:  # matplotlib missing
        print("matplotlib not available: no plots")
        return None


def _save(plt, fig, out_dir: str, stem: str) -> None:
    fig.tight_layout()
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(out_dir, f"{stem}.{ext}"), dpi=150)
    plt.close(fig)
    print(f"wrote {os.path.join(out_dir, stem)}.png/.pdf")


def plot_matched(rows: List[dict], out_dir: str, N: int, K_list: Sequence[int],
                 convention: str) -> None:
    plt = _pyplot()
    if plt is None:
        return
    for K in K_list:
        sel = [r for r in rows if r["protocol"] == "matched" and r["K"] in (0, K)]
        methods: List[str] = []
        for r in sel:
            if r["method"] not in methods:
                methods.append(r["method"])
        if len(methods) <= 2:
            continue
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4.2))
        for m in methods:
            pts = sorted((r["snr_test_db"], r["mse"], r["gain_vs_map_pct"])
                         for r in sel if r["method"] == m)
            lab = next(r["label"] for r in sel if r["method"] == m)
            x = [a[0] for a in pts]
            ax1.semilogy(x, [a[1] for a in pts], label=lab, lw=1.2, ms=4, **STYLE.get(m, {}))
            if m != "MAP":
                ax2.plot(x, [a[2] for a in pts], label=lab, lw=1.2, ms=4, **STYLE.get(m, {}))
        ax1.axhline(PRIOR_VAR, color="gray", ls=":", lw=1.0, label="prior variance")
        ax1.set_xlabel(f"SNR [dB] ({convention})")
        ax1.set_ylabel("MSE")
        ax1.set_title(f"N = {N}, K = {K}: matched training/test SNR")
        ax1.grid(True, which="both", alpha=0.3)
        ax1.legend(fontsize=8)
        ax2.axhline(0.0, color="tab:red", lw=1.0)
        ax2.set_xlabel(f"SNR [dB] ({convention})")
        ax2.set_ylabel("MSE reduction relative to MAP [%]")
        ax2.grid(True, alpha=0.3)
        ax2.legend(fontsize=8)
        _save(plt, fig, out_dir, f"matched_N{N}_K{K}")


def plot_mismatch(rows: List[dict], out_dir: str, N: int, K_list: Sequence[int],
                  train_snrs: Sequence[float], convention: str) -> None:
    plt = _pyplot()
    if plt is None:
        return
    mm = [r for r in rows if r["protocol"] == "mismatch"]
    ref = {m: sorted((r["snr_test_db"], r["mse"]) for r in mm if r["method"] == m)
           for m in ("MAP", "MMSE")}
    for K in K_list:
        methods: List[str] = []
        for r in mm:
            if r["K"] == K and r["method"] not in methods:
                methods.append(r["method"])
        if not methods:
            continue
        fig, axes = plt.subplots(1, len(methods), figsize=(5.4 * len(methods), 4.3),
                                 squeeze=False)
        for ax, m in zip(axes[0], methods):
            for name in ("MAP", "MMSE"):
                ax.semilogy([a for a, _ in ref[name]], [b for _, b in ref[name]], label=name,
                            lw=1.3, color=STYLE[name]["color"], ls=STYLE[name]["ls"])
            for i, st in enumerate(train_snrs):
                pts = sorted((r["snr_test_db"], r["mse"]) for r in mm
                             if r["method"] == m and r["K"] == K and r["snr_train_db"] == st)
                if not pts:
                    continue
                col = TRAIN_COLORS[i % len(TRAIN_COLORS)]
                ax.semilogy([a for a, _ in pts], [b for _, b in pts], ls="--", marker="o",
                            ms=3, lw=1.0, color=col, label=f"trained at {st:g} dB")
                at = [b for a, b in pts if abs(a - st) < 1e-9]
                if at:
                    ax.semilogy([st], at, "s", ms=9, mfc="none", mec=col, mew=1.5)
            ax.set_title(next(r["label"] for r in mm if r["method"] == m and r["K"] == K))
            ax.set_xlabel(f"test SNR [dB] ({convention})")
            ax.set_ylabel("MSE")
            ax.grid(True, which="both", alpha=0.3)
            ax.legend(fontsize=7)
        _save(plt, fig, out_dir, f"mismatch_N{N}_K{K}")


# ======================================================================================
# Command line
# ======================================================================================
def build_parser() -> argparse.ArgumentParser:
    ap = argparse.ArgumentParser(
        description="Uhlich (2015) versus the zero-mean perturbation bank of the DSP2026 "
                    "manuscript for single-tone frequency estimation.",
        formatter_class=argparse.ArgumentDefaultsHelpFormatter)
    ap.add_argument("command", choices=["calibrate", "matched", "mismatch", "all"])
    ap.add_argument("--preset", choices=sorted(PRESETS), default="uhlich")
    g = ap.add_argument_group("model and SNR")
    g.add_argument("--N", type=int, default=None, help="observation length")
    g.add_argument("--A", type=float, default=1.0, help="amplitude")
    g.add_argument("--snr-convention", choices=sorted(SNR_CONVENTIONS), default=None)
    g.add_argument("--snrs", type=float, nargs="+", default=None,
                   help="SNR grid in dB (matched: train = test; mismatch: test grid)")
    g.add_argument("--train-snrs", type=float, nargs="+", default=None,
                   help="training SNRs of the mismatch protocol")
    g.add_argument("--calib-snrs", type=float, nargs="+", default=[-10.0, 5.0, 20.0])
    g = ap.add_argument_group("data")
    g.add_argument("--M-train", type=int, default=None, help="training pairs (Uhlich: I = 1e5)")
    g.add_argument("--M-val", type=int, default=None, help="validation pairs M'")
    g.add_argument("--M-test", type=int, default=None, help="test trials")
    g.add_argument("--seed", type=int, default=2026)
    g = ap.add_argument_group("estimators")
    g.add_argument("--K", type=int, nargs="+", default=None, help="bank sizes")
    g.add_argument("--methods", nargs="+", default=None,
                   help=f"subset of {list(ALL_METHODS)} or 'all'; default: all four for "
                        f"`matched`, Uhlich and Proposed for `mismatch` (MAP/MMSE always)")
    g.add_argument("--n-grid", type=int, default=1024, help="N_theta of the MAP grid")
    g.add_argument("--newton-iters", type=int, default=3, help="T")
    g.add_argument("--refine-hessian", choices=["gn", "full"], default="gn")
    g.add_argument("--ift-hessian", choices=["full", "gn"], default="full")
    g.add_argument("--ift-floor", type=float, default=0.05,
                   help="curvature floor, fraction of Gauss-Newton curvature (0 = exact)")
    g.add_argument("--n-quad", type=int, default=4096, help="N_q of the MMSE quadrature")
    g = ap.add_argument_group("Algorithm 1 (Proposed)")
    g.add_argument("--beta", type=float, default=4.0)
    g.add_argument("--alphas", type=float, nargs="+", default=[0.1, 0.5, 1.0, 2.0, 3.0, 4.0])
    g.add_argument("--eta0", type=float, default=0.5, help="initial step in units of sigma_v")
    g.add_argument("--eta-min", type=float, default=1e-3)
    g.add_argument("--T-r", type=int, default=50, help="maximum refinement iterations")
    g.add_argument("--grad-scaling", choices=["max-entry", "max-row"], default="max-entry")
    g.add_argument("--kappa", type=float, default=0.02)
    g.add_argument("--t-threshold", type=float, default=3.0)
    g.add_argument("--no-gate", action="store_true", help="no validation gate for Proposed")
    g.add_argument("--gate-all", action="store_true", help="validation gate for every method")
    g = ap.add_argument_group("Uhlich's formulation")
    g.add_argument("--uhlich-solver", choices=["lbfgs", "ipopt"], default="lbfgs")
    g.add_argument("--lbfgs-iter", type=int, default=100)
    g.add_argument("--lbfgs-history", type=int, default=20)
    g.add_argument("--ipopt-iter", type=int, default=200)
    g.add_argument("--n-starts", type=int, default=2)
    g = ap.add_argument_group("run")
    g.add_argument("--device", default="auto", help="auto, cpu, cuda, cuda:1, mps, ...")
    g.add_argument("--dtype", choices=["float64", "float32"], default="float64")
    g.add_argument("--grid-dtype", choices=["float32", "float64"], default="float32")
    g.add_argument("--chunk", type=int, default=4096, help="observations per batch")
    g.add_argument("--threads", type=int, default=0, help="CPU threads (0 = PyTorch default)")
    g.add_argument("--out-dir", default="results_ne_vs_uhlich")
    g.add_argument("--no-plot", action="store_true")
    g.add_argument("--quick", action="store_true", help="small smoke-test configuration")
    return ap


def apply_preset(args) -> None:
    P = PRESETS[args.preset]
    user_snrs, user_train = args.snrs is not None, args.train_snrs is not None
    for key in ("N", "snr_convention", "snrs", "train_snrs", "K", "M_train", "M_val", "M_test"):
        if getattr(args, key) is None:
            setattr(args, key, P[key])
    if args.quick:
        args.M_train = min(args.M_train, 3000)
        args.M_val = min(args.M_val, 3000)
        args.M_test = min(args.M_test, 5000)
        args.T_r, args.lbfgs_iter = min(args.T_r, 5), min(args.lbfgs_iter, 10)
        args.ipopt_iter, args.n_starts = min(args.ipopt_iter, 20), 1
        args.n_quad = min(args.n_quad, 2048)
        if not user_snrs:
            args.snrs = list(P["quick_snrs"])
        if not user_train:
            args.train_snrs = list(P["quick_train_snrs"])
    if args.methods is None:
        args.methods_matched = list(ALL_METHODS)
        args.methods_mismatch = ["Uhlich", "Proposed"]
    else:
        chosen = list(ALL_METHODS) if "all" in args.methods else list(args.methods)
        unknown = [m for m in chosen if m not in ALL_METHODS]
        if unknown:
            raise SystemExit(f"unknown method(s) {unknown}; choose from {list(ALL_METHODS)}")
        args.methods_matched = args.methods_mismatch = chosen
    if any(k < 1 for k in args.K):
        raise SystemExit("--K values must be >= 1")
    args.Delta = 2.0 * math.pi / args.N        # anomaly threshold of Algorithm 1


def main(argv: Optional[Sequence[str]] = None) -> None:
    args = build_parser().parse_args(argv)
    apply_preset(args)
    if args.threads > 0:
        torch.set_num_threads(args.threads)
    device = ("cuda" if torch.cuda.is_available() else "cpu") if args.device == "auto" \
        else args.device
    dtype = torch.float64 if args.dtype == "float64" else torch.float32
    grid_dtype = torch.float32 if args.grid_dtype == "float32" else torch.float64
    if str(device).startswith("mps") and torch.float64 in (dtype, grid_dtype):
        print("note: MPS has no float64 support; using float32 throughout")
        dtype = grid_dtype = torch.float32
    torch.set_float32_matmul_precision("highest")   # no TF32 in the grid correlations
    model = ToneModel(args.N, args.A, args.n_grid, args.newton_iters, args.refine_hessian,
                      args.ift_hessian, args.ift_floor, device, dtype, grid_dtype, args.chunk)
    os.makedirs(args.out_dir, exist_ok=True)

    print(f"N = {args.N}, A = {args.A}, SNR convention '{args.snr_convention}' "
          f"(sigma_v^2 = {SNR_CONVENTIONS[args.snr_convention]:g} A^2 10^(-SNR/10); "
          f"manuscript gamma = SNR {gamma_offset_db(args.snr_convention):+.2f} dB), "
          f"device {device}")
    print(f"M_train = {args.M_train}, M_val = {args.M_val}, M_test = {args.M_test}, "
          f"N_theta = {args.n_grid}, T = {args.newton_iters}, N_q = {args.n_quad}, "
          f"Delta = 2 pi/N = {args.Delta:.4f}")
    print(f"online cost (manuscript Table 1): MAP {online_ops(args.N, args.n_grid, 0, args.newton_iters)}; "
          + ", ".join(f"K = {K}: {online_ops(args.N, args.n_grid, K, args.newton_iters)}"
                      for K in args.K) + " (identical for Uhlich and Proposed at equal K)")
    if args.command in ("matched", "mismatch", "all"):
        methods = set(args.methods_matched) | set(args.methods_mismatch)
        for K in args.K:
            for m in sorted(methods):
                ok, why = applicable(m, K)
                if not ok:
                    print(f"note: {m} skipped for K = {K}: {why}")

    test = make_split(model, args.M_test, args.seed + 3)
    splits = {"test": test}
    if args.command in ("matched", "mismatch", "all"):
        splits["train"] = make_split(model, args.M_train, args.seed + 1)
        splits["val"] = make_split(model, args.M_val, args.seed + 2)
    tag = f"N{args.N}"
    save_json(os.path.join(args.out_dir, f"config_{args.command}_{tag}.json"), vars(args))

    if args.command in ("calibrate", "all"):
        write_csv(os.path.join(args.out_dir, "calibration.csv"), run_calibrate(args, model, test))
    if args.command in ("matched", "all"):
        rows, pairs, banks = run_matched(args, model, splits)
        write_csv(os.path.join(args.out_dir, f"matched_{tag}.csv"), rows)
        write_csv(os.path.join(args.out_dir, f"matched_pairs_{tag}.csv"), pairs)
        save_json(os.path.join(args.out_dir, f"banks_matched_{tag}.json"), banks)
        print_matched_summary(rows, pairs, args.K)
        if not args.no_plot:
            plot_matched(rows, args.out_dir, args.N, args.K, args.snr_convention)
    if args.command in ("mismatch", "all"):
        rows, banks = run_mismatch(args, model, splits)
        write_csv(os.path.join(args.out_dir, f"mismatch_{tag}.csv"), rows)
        save_json(os.path.join(args.out_dir, f"banks_mismatch_{tag}.json"), banks)
        if not args.no_plot:
            plot_mismatch(rows, args.out_dir, args.N, args.K, args.train_snrs,
                          args.snr_convention)
    print(f"\nresults in {os.path.abspath(args.out_dir)}")


if __name__ == "__main__":
    main(["all", "--preset", "dsp"])

N = 32, A = 1.0, SNR convention 'manuscript' (sigma_v^2 = 0.5 A^2 10^(-SNR/10); manuscript gamma = SNR -0.00 dB), device cuda
M_train = 100000, M_val = 20000, M_test = 100000, N_theta = 1024, T = 3, N_q = 4096, Delta = 2 pi/N = 0.1963
online cost (manuscript Table 1): MAP 33891; K = 8: 41752 (identical for Uhlich and Proposed at equal K)
wrote results_ne_vs_uhlich\config_all_N32.json

[calibrate] Uhlich (2015), N = 5: MAP 0.41 and MMSE 0.29 at '5 dB' (text); about 1.6 / 0.8 at -10 dB and 1e-3 at 20 dB (Fig. 5)
  note: N = 32, while Uhlich's values refer to N = 5
  manuscript  -10.0 dB  sigma_v^2 =    5.0000  MAP 1.0103  MMSE 0.6299
  manuscript   +5.0 dB  sigma_v^2 =    0.1581  MAP 0.0000  MMSE 0.0000
  manuscript  +20.0 dB  sigma_v^2 =    0.0050  MAP 0.0000  MMSE 0.0000
  unit        -10.0 dB  sigma_v^2 =   10.0000  MAP 1.3058  MMSE 0.7483
  unit         +5.0 dB  sigma_v^2 =    0.3162  MAP 0.0001  MMSE 0.0001
  unit        +20.0 dB  sigma_v^2 =    0.0100  MAP 0.0000  MMSE 0.0000
  uhl